In [1]:
import pandas as pd
import pandas_gbq
import plotly.express as px
import sys, requests
from pathlib import Path
from zipfile import ZipFile
import urllib
from PIL import Image

In [2]:
#Change the date to the previous day
query = """
SELECT * FROM `perceptive-ivy-290216.f1_api.results_race`
WHERE YEAR=2024
"""
project_id = "perceptive-ivy-290216"
df_bq2 = pandas_gbq.read_gbq(query, project_id=project_id, dialect='standard')
df_bq2=df_bq2.sort_values(by=["TeamName"])

Downloading: 100%|██████████|


In [3]:
df_bq2.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Year,GP
329,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,...,9,9.0,NaT,NaT,NaT,NaT,+1 Lap,2.0,2024,Dutch Grand Prix
321,10,P GASLY,GAS,gasly,Alpine,ff87bc,alpine,Pierre,Gasly,Pierre Gasly,...,18,20.0,NaT,NaT,NaT,NaT,+1 Lap,0.0,2024,Bahrain Grand Prix
322,10,P GASLY,GAS,gasly,Alpine,ff87bc,alpine,Pierre,Gasly,Pierre Gasly,...,R,18.0,NaT,NaT,NaT,NaT,Gearbox,0.0,2024,Saudi Arabian Grand Prix
164,31,E OCON,OCO,ocon,Alpine,0093cc,alpine,Esteban,Ocon,Esteban Ocon,...,14,15.0,NaT,NaT,NaT,NaT,+1 Lap,0.0,2024,Italian Grand Prix
33,31,E OCON,OCO,ocon,Alpine,0093cc,alpine,Esteban,Ocon,Esteban Ocon,...,15,0.0,NaT,NaT,NaT,NaT,+1 Lap,0.0,2024,Azerbaijan Grand Prix


In [53]:
df_bq=df_bq2.groupby(['TeamName','Year'])['Points'].agg('sum').reset_index()
df_bq=df_bq.sort_values(by='Points', ascending=False)
Year='2024'
df_bq

,TeamName,Year,Points
5,McLaren,2024,609.0
2,Ferrari,2024,595.0
8,Red Bull Racing,2024,537.0
6,Mercedes,2024,433.0
1,Aston Martin,2024,94.0
0,Alpine,2024,63.0
3,Haas F1 Team,2024,51.0
7,RB,2024,40.0
9,Williams,2024,17.0
4,Kick Sauber,2024,4.0


In [68]:
fig = px.histogram(
  df_bq, 
  x="TeamName", 
  y="Points",
  title="<b>Constructor Points for the {} Season</b>".format(Year),
  template="plotly_white",
  # hover_data=['Conference','Record','ConferenceRecord','DivisionRecord', 'WINS', 'LOSSES', 'WinPCT', 'HOME', 'ROAD', 'L10', 'OT', 
  #             'ThreePTSOrLess', 'TenPTSOrMore', 'LongHomeStreak','LongRoadStreak','LongWinStreak', 'LongLossStreak','AheadAtHalf', 'BehindAtHalf', 'TiedAtHalf', 'AheadAtThird',
  #             'BehindAtThird', 'TiedAtThird', 'Score100PTS', 'OppScore100PTS','OppOver500', 'LeadInFGPCT', 'LeadInReb', 'FewerTurnovers', 'PreAS', 'PostAS', 'Season'], 
  height=700, 
  width=1200,
  color="TeamName",
  color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"}
  )

for x,y in zip(df_bq.TeamName, df_bq.Points):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_Scripts/Team_Logos").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig.add_layout_image(
          x=x,
          y=y+50,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=75,
          sizey=75,
          xanchor="center",
          yanchor="middle",
      )

fig.update_layout(
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis_range=[0,700],
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig.update_layout(showlegend=False)
fig

In [ ]:
save_year=2024

In [22]:
# fig.write_html("/Users/rdesh723/statpulse-html/plots/Standings/{}/NBA Standings For The {} Season.html".format(save_year,year),full_html=False, include_plotlyjs='cdn')